In [2]:
import pandas as pd 


In [3]:
df = pd.read_csv("/Users/tam_pr/tcg_bot/data/raw/EN_Card_Data_R2_full.csv")

In [5]:
columns = df.columns
columns

Index(['Card ID', 'Card Name', 'Expansion', 'Collection No.',
       'Stage (Pokémon)/Type (Energy and Trainer)', 'Rule', 'Category',
       'Previous stage', 'HP', 'Type', 'Weakness', 'Resistance (Type)',
       'Retreat', 'Move Name', 'Cost', 'Damage', 'Effect Explanation',
       'Species'],
      dtype='str')

In [12]:
df.shape

(2273, 18)

In [6]:
rename_map = {
    "Card ID": "card_id",
    "Card Name": "card_name",
    "Expansion": "expansion",
    "Collection No.": "collection_no",
    "Stage (Pokémon)/Type (Energy and Trainer)": "card_kind",
    "Rule": "rule",
    "Category": "category",
    "Previous stage": "previous_stage",
    "HP": "hp",
    "Type": "type",
    "Weakness": "weakness",
    "Resistance (Type)": "resistance",
    "Retreat": "retreat_cost",
    "Move Name": "move_name",
    "Cost": "cost",
    "Damage": "damage",
    "Effect Explanation": "effect_text",
    "Species": "species",
}

df = df.rename(columns=rename_map)

In [13]:
print(df.columns.tolist())
assert len(df.columns) == 18
assert df.columns.is_unique
assert not any(c != c.lower() or " " in c for c in df.columns)


['card_id', 'card_name', 'expansion', 'collection_no', 'card_kind', 'rule', 'category', 'previous_stage', 'hp', 'type', 'weakness', 'resistance', 'retreat_cost', 'move_name', 'cost', 'damage', 'effect_text', 'species']


In [11]:
df.shape

(2273, 18)

### Data Sectioning

In Pokémon TCG there is three kinds of primary cards: <br>

1. Trainer cards: There's various kinds: Items, Supporters, Stadiums and Pokémon Tools. 
2. Pokémon cards: There's three kinds: Basic, Stage 1 and Stage 2. 
3. Energy cards: Powers up Pokémons, there's 11 different kinds. 

The dataset we are working on contains all the different kinds of cards, we need to section them in this three main groups so we can process each one with the most optimal strategy. 

<br>

The following sub-datasets will be obtained in this first cut, to divide the kinds of cards 

| # | Table           | Row =               | Key       |
|---|-----------------|---------------------|-----------|
| 1 | `cards`         | one card (any kind) | `card_id` |
| 2 | `pokemon_cards` | one Pokémon card    | `card_id` |
| 3 | `trainer_cards` | one Trainer card    | `card_id` |


In [14]:
print(df["card_kind"].value_counts(dropna=False))

card_kind
Basic Pokémon      1073
Stage 1 Pokémon     699
Stage 2 Pokémon     258
Item                 90
Supporter            69
Pokémon Tool         30
Stadium              29
Special Energy       17
Basic Energy          8
Name: count, dtype: int64


In [15]:
pokemon_cards = ["Basic Pokémon", "Stage 1 Pokémon", "Stage 2 Pokémon"]
trainer_cards = ["Item", "Supporter", "Pokémon Tool", "Stadium"]
energy_cards = ["Basic Energy", "Special Energy"]

In [16]:
pokemon = df[df["card_kind"].isin(pokemon_cards)]
trainer = df[df["card_kind"].isin(trainer_cards)]
energy  = df[df["card_kind"].isin(energy_cards)]

In [17]:
print(len(pokemon), len(trainer), len(energy))
assert len(pokemon) + len(trainer) + len(energy) == len(df)

2030 218 25


In [20]:
assert not (set(pokemon.index) & set(trainer.index))
assert not (set(pokemon.index) & set(energy.index))
assert not (set(trainer.index) & set(energy.index))

In [21]:
leftover = df[~df.index.isin(pokemon.index.union(trainer.index).union(energy.index))]
print(len(leftover)) 

0


In [22]:
for name, part in [("pokemon", pokemon), ("trainer", trainer), ("energy", energy)]:
    print(f"\n{name}")
    print(part.replace("n/a", pd.NA).notna().sum())


pokemon
card_id           2030
card_name         2030
expansion         2021
collection_no     2030
card_kind         2030
rule               355
category           376
previous_stage     957
hp                2030
type              2030
weakness          1953
resistance         443
retreat_cost      1969
move_name         2030
cost              1754
damage            1446
effect_text       1436
species            206
dtype: int64

trainer
card_id           218
card_name         218
expansion         217
collection_no     218
card_kind         218
rule               26
category           20
previous_stage     14
hp                 14
type                0
weakness            0
resistance          0
retreat_cost        0
move_name           8
cost                1
damage              1
effect_text       218
species             0
dtype: int64

energy
card_id           25
card_name         25
expansion         25
collection_no     25
card_kind         25
rule               3
category    

In [23]:
pokemon.head(10)

,card_id,card_name,expansion,collection_no,card_kind,rule,category,previous_stage,hp,type,weakness,resistance,retreat_cost,move_name,cost,damage,effect_text,species
20,21,Scrafty,PROMO,188,Stage 1 Pokémon,NaN,NaN,Scraggy,120.0,{D},{G},NaN,1.0,Nab ’n’ Dash,●,NaN,Search your deck for a number of cards up to t...,NaN
21,21,Scrafty,PROMO,188,Stage 1 Pokémon,NaN,NaN,Scraggy,120.0,{D},{G},NaN,1.0,High Jump Kick,{D}●●,100,NaN,NaN
22,22,Hippopotas,DRI,105,Basic Pokémon,NaN,NaN,NaN,90.0,{F},{G},NaN,3.0,Push Down,{F},10,Switch out your opponent’s Active Pokémon to t...,NaN
23,23,Hippowdon,DRI,106,Stage 1 Pokémon,NaN,NaN,Hippopotas,160.0,{F},{G},NaN,4.0,Ram,{F}{F},60,NaN,NaN
24,23,Hippowdon,DRI,106,Stage 1 Pokémon,NaN,NaN,Hippopotas,160.0,{F},{G},NaN,4.0,Super Sandstorm,{F}{F}●,150,This attack also does 40 damage to each Benche...,NaN
25,24,Team Rocket's Kangaskhan ex,ASC,162,Basic Pokémon,Pokémon ex,Trainer's Pokémon(Team Rocket),NaN,230.0,{C},{F},NaN,2.0,\r\nComet Punch,●●,30×,Flip 4 coins. This attack does 30 damage for e...,NaN
26,24,Team Rocket's Kangaskhan ex,ASC,162,Basic Pokémon,Pokémon ex,Trainer's Pokémon(Team Rocket),NaN,230.0,{C},{F},NaN,2.0,Wicked Impact,●●●,120,If you played a Supporter card that has “Team ...,NaN
27,25,Pinsir,TWM,3,Basic Pokémon,NaN,NaN,NaN,110.0,{G},{R},NaN,2.0,Slow Crunch,{G}●,NaN,Discard all Energy from this Pokémon. At the e...,NaN
28,25,Pinsir,TWM,3,Basic Pokémon,NaN,NaN,NaN,110.0,{G},{R},NaN,2.0,Superpowered Horns,{G}●●,100,NaN,NaN
29,26,Leafeon,TWM,11,Stage 1 Pokémon,NaN,NaN,Eevee,120.0,{G},{R},NaN,1.0,Leaflet Blessings,●,NaN,Attach a Basic {G} Energy card from your hand ...,NaN
